In [43]:
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

BASE_DIR = Path.cwd().parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

df = pd.read_csv(DATA_PROCESSED / "events_with_car.csv", sep=";")

print("Loaded:", df.shape)
df.head()

Loaded: (72, 18)


,event_id,event_date,trading_date,ticker,publisher,studio,is_rockstar,game,franchise,event_type,sentiment,impact_expectation_manual,adj_close,return,market_return,AR_event,CAR_m1_p1,CAR_m5_p5
0,ATVI_2019_CODMOBILE_LAUNCH,2019-10-01,2019-10-01,ATVI,Activision,TiMi Studios,0,Call of Duty: Mobile,Call of Duty,Release,Positive,Medium,94.157463,-0.010938,-0.012258,-0.001198,0.005278,-0.013979
1,ATVI_2019_CODMW_RELEASE,2019-10-25,2019-10-25,ATVI,Activision,Infinity Ward,0,Call of Duty: Modern Warfare,Call of Duty,Release,Positive,High,93.729248,0.003438,0.004073,-0.000318,0.000526,-0.002328
2,ATVI_2020_WARCRAFT3_REFORGED,2020-01-28,2020-01-28,ATVI,Activision Blizzard,Blizzard,0,Warcraft III: Reforged,Warcraft,Controversy,Negative,Medium,108.901489,0.012120,0.010054,0.003422,0.003371,-0.024705
3,ATVI_2020_WARZONE_LAUNCH,2020-03-10,2020-03-10,ATVI,Activision,Infinity Ward,0,Call of Duty: Warzone,Call of Duty,Release,Positive,High,100.609787,0.024274,0.049396,-0.016937,0.001569,-0.031189
4,ATVI_2021_LAWSUIT,2021-07-20,2021-07-20,ATVI,Activision Blizzard,NaN,0,NaN,Activision,Controversy,negative,high,137.807159,-0.001204,0.015163,-0.014124,-0.024080,0.003237


In [39]:
# Normalize categorical columns
text_cols = ["publisher", "studio", "event_type", "sentiment", "franchise"]

for col in text_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("nan", "")
    )

In [40]:
# GTA dummy
df["franchise_gta"] = (df["franchise"] == "gta").astype(int)

# Force numeric types
num_cols = ["CAR_m1_p1", "AR_event", "market_return", "is_rockstar", "franchise_gta"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows where CAR_m1_p1 is missing (no dependent variable)
df = df.dropna(subset=["CAR_m1_p1"])
print("After dropping Na CAR rows:", df.shape)

After dropping Na CAR rows: (72, 19)


In [41]:
# Dependent variable
y = df["CAR_m1_p1"]

# Feature set
feature_cols = [
    "franchise_gta",
    "is_rockstar",
    "market_return",
    "AR_event",
    "event_type",
    "publisher",
    "sentiment",
]

# Build X with dummies for categoricals
X_base = df[feature_cols]

X = pd.get_dummies(
    X_base,
    columns=["event_type", "publisher", "sentiment"],
    drop_first=True
)

# Make sure EVERYTHING is numeric
X = X.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(y, errors="coerce")

# Align and drop any remaining NaNs
mask = y.notna() & X.notna().all(axis=1)
X = X.loc[mask]
y = y.loc[mask]

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)
print("Any object dtypes left?", (X.dtypes == "object").any())

Final X shape: (68, 20)
Final y shape: (68,)
Any object dtypes left? False


In [46]:
# --------------------------
X = sm.add_constant(X)
# ensure numeric numpy arrays to avoid pandas/object dtype issues
ols_model = sm.OLS(y.values.astype(float), X.values.astype(float)).fit()

print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.484
Model:                            OLS   Adj. R-squared:                  0.294
Method:                 Least Squares   F-statistic:                     2.551
Date:                Mon, 24 Nov 2025   Prob (F-statistic):            0.00487
Time:                        10:37:41   Log-Likelihood:                 145.43
No. Observations:                  68   AIC:                            -252.9
Df Residuals:                      49   BIC:                            -210.7
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0156      0.030      0.517      0.6